In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
import pickle

import pandas as pd
import category_encoders as ce
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OneHotEncoder
from common import correlation_based_feature_selection as cbfs

In [ ]:
pip freeze > requirements.txt

In [ ]:
# Read in data
clinical_features = pd.read_pickle("./data/clinical_features.pkl")
print(clinical_features.shape)
Resnet_features = pd.read_pickle("./data/Resnet_features.pkl")
print(Resnet_features.shape)

# 1. Input Features

## 1.1 Clinical Features

In [ ]:
X_clinical = clinical_features.copy()
X_clinical = X_clinical.drop(
    columns=["Progression Free Survival", "Event"]
)

# apply label encoding
X_clinical = X_clinical.replace(
    to_replace={
        "Sex": {
            "Female": 0,
            "Male": 1,
        },
        "Extent of Tumor Resection": {
            "Not Applicable": 0,
            "Unavailable": 0,
            "Biopsy only": 1,
            "Partial resection": 2,
            "Gross/Near total resection": 3,
            
        },
        "Chemotherapy": {
            "Yes": 1,
            "No": 0,
            "Not Applicable": 0,
            "Not Reported": 0,
            "Unavailable": 0,
        },
        "Radiation": {
            "Yes": 1,
            "No": 0,
            "Not Applicable": 0,
            "Not Reported": 0,
            "Unavailable": 0,
        },
    }
)


X_clinical["NF1"] = X_clinical["NF1"].apply(
    lambda x: 1 if x == "Neurofibromatosis, Type 1 (NF-1)" else 0
)

# split subjects into Discovery and Replicate cohorts
discovery_clinical = X_clinical[X_clinical["Cohort"] == "Discovery"].copy()
replicate_clinical = X_clinical[X_clinical["Cohort"] == "Replicate"].copy()


# apply count encoding
encoder = ce.CountEncoder()
columns = [
    "Tumor Location",
]
encoder.fit(discovery_clinical[columns])
discovery_clinical[columns] = encoder.transform(discovery_clinical[columns])
replicate_clinical[columns] = encoder.transform(replicate_clinical[columns])
with open("./encoder.pkl", mode="wb") as file:
    pickle.dump(encoder, file)

# apply standardization
scaler = StandardScaler()
columns = [
    "Age at Diagnosis",
    "Tumor Location",
]
scaler.fit(discovery_clinical[columns])
discovery_clinical[columns] = scaler.transform(discovery_clinical[columns])
replicate_clinical[columns] = scaler.transform(replicate_clinical[columns])
with open("./scaler_clinical.pkl", mode="wb") as file:
    pickle.dump(scaler, file)

# apply normalization
normalizer = 3
discovery_clinical["Extent of Tumor Resection"] = discovery_clinical[
    "Extent of Tumor Resection"
].apply(lambda x: x / normalizer)
replicate_clinical["Extent of Tumor Resection"] = replicate_clinical[
    "Extent of Tumor Resection"
].apply(lambda x: x / normalizer)

# concatenate cohorts
X_clinical = pd.concat([discovery_clinical, replicate_clinical])

# cache clinical input features
X_clinical.to_pickle("X_clinical.pkl")
X_clinical.shape

In [ ]:
X_clinical

## 1.2 Radiomic Features

In [ ]:
X_Resnet = Resnet_features.copy()
X_Resnet = X_Resnet.drop(columns=["Session"])
# split subjects into Discovery and Replicate cohorts
discovery_resnet = X_Resnet[X_Resnet.index.isin(discovery_clinical.index)].copy()
replicate_resnet = X_Resnet[X_Resnet.index.isin(replicate_clinical.index)].copy()

# apply imputing
imputer = SimpleImputer()
imputer.fit(discovery_resnet)
discovery_resnet.loc[:, :] = imputer.transform(discovery_resnet)
replicate_resnet.loc[:, :] = imputer.transform(replicate_resnet)
with open("./imputer.pkl", mode="wb") as file:
    pickle.dump(imputer, file)

# apply standardization
scaler = StandardScaler()
scaler.fit(discovery_resnet)
discovery_resnet.loc[:, :] = scaler.transform(discovery_resnet)
replicate_resnet.loc[:, :] = scaler.transform(replicate_resnet)
with open("./scaler_radiomic.pkl", mode="wb") as file:
    pickle.dump(scaler, file)

# remove constant features
remover = VarianceThreshold(threshold=0)
remover.fit(discovery_resnet)
column_indices = list(remover.get_support(indices=True))
discovery_resnet = discovery_resnet[discovery_resnet.columns[column_indices]]
replicate_resnet = replicate_resnet[replicate_resnet.columns[column_indices]]
with open("./remover.pkl", mode="wb") as file:
    pickle.dump(remover, file)

# remove correlated features (|r| > 0.9)
dropper = cbfs.main(discovery_resnet)
discovery_resnet = discovery_resnet.drop(columns=dropper)
replicate_resnet = replicate_resnet.drop(columns=dropper)
with open("./dropper.pkl", mode="wb") as file:
    pickle.dump(dropper, file)

# concatenate cohorts
X_resnet = pd.concat([discovery_resnet, replicate_resnet]).copy()
X_resnet["Cohort"] = X_resnet.index.map(
    lambda x: "Discovery" if x in discovery_clinical.index else "Replicate"
)

# cache radiomic features
X_resnet.to_pickle("X_resnet.pkl")
X_resnet.shape

In [ ]:
X_resnet

# 2. Output Features

In [ ]:
# select columns
y = clinical_features[["Progression Free Survival", "Event"]].merge(
    X_clinical["Cohort"].to_frame(), left_index=True, right_index=True
)
# compute age in months
y["Progression Free Survival"] = y["Progression Free Survival"].apply(
    lambda x: int(x) / 30.417
)

# cache output features
y.to_pickle("./y.pkl")
y.shape